# 03_載入TM模型_鏡頭辨識.py
**電腦視覺與深度學習 › Day18 CNN + Teachable Machine + MediaPipe**

這支程式把你在 Teachable Machine 網頁上訓練好的模型接上鏡頭,即時判斷畫面是哪一類。從 Day14 的「自己寫規則」,換成「讓模型自己學」。

> 🎧 有逐段旁白、逐句拆解的版本在 [pyl Academy](https://friencdking0000.github.io/pyl-academy/#/learn/PY-CVAI/4/0)。
>
> **怎麼用**:由上往下,每格按左邊 ▶ 執行;或選單「執行階段 › 全部執行」。
>
> ⚡ 這支會用到深度學習,建議先到「執行階段 › 變更執行階段類型」選 **T4 GPU**。
>
> 📷 Colab 沒有攝影機、也開不了視窗:下面會**改用範例圖片**代替鏡頭,影像直接顯示在格子底下(只顯示前幾張)。

In [ ]:
#@title 0. 準備環境(下載課程範例檔、安裝套件)
!git clone -q --depth 1 https://github.com/friencdking0000/2026AI-STUST.git /content/2026AI-STUST 2>/dev/null; echo 課程檔案已就緒
%cd "/content/2026AI-STUST/Week2/DAY9"
import sys, os; sys.argv = sys.argv[:1]  # 讓 argparse / sys.argv 在 notebook 裡也正常
__file__ = os.path.join(os.getcwd(), '03_載入TM模型_鏡頭辨識.py')  # notebook 沒有 __file__,程式靠它找同資料夾的檔案

### 需要你自己的模型
這支要用你在 Teachable Machine 匯出的 `keras_model.h5`、`labels.txt`。執行下一格後按「選擇檔案」上傳。

In [ ]:
#@title 上傳模型檔
from google.colab import files
files.upload()

In [ ]:
#@title 1. Colab 沒有攝影機和視窗:改用範例圖片、影像直接顯示在下面
import cv2, numpy as np, urllib.request, os
from google.colab.patches import cv2_imshow
def _pyl_sample():
    if not os.path.exists('/content/_sample.jpg'):
        urllib.request.urlretrieve('https://ultralytics.com/images/bus.jpg', '/content/_sample.jpg')
    return cv2.imread('/content/_sample.jpg')
class _PylCam:
    def __init__(self, *a, **k): self.n = 0; self.img = _pyl_sample()
    def isOpened(self): return True
    def read(self):
        self.n += 1
        return (True, self.img.copy()) if self.n <= 30 else (False, None)
    def set(self, *a): return True
    def get(self, *a): return 0
    def release(self): pass
_pyl_VC = cv2.VideoCapture
cv2.VideoCapture = lambda src=0, *a, **k: _PylCam() if isinstance(src, int) else _pyl_VC(src, *a, **k)
_pyl_shown = [0]
def _pyl_imshow(name, img):
    _pyl_shown[0] += 1
    if _pyl_shown[0] <= 6:
        print('▼', name); cv2_imshow(img)
cv2.imshow = _pyl_imshow
cv2.waitKey = lambda *a: ord('q') if _pyl_shown[0] >= 6 else -1
cv2.destroyAllWindows = lambda *a: None
cv2.namedWindow = lambda *a, **k: None
cv2.createTrackbar = lambda *a, **k: None
cv2.getTrackbarPos = lambda *a, **k: 0
cv2.setMouseCallback = lambda *a, **k: None
cv2.resizeWindow = cv2.moveWindow = cv2.setWindowProperty = cv2.setWindowTitle = lambda *a, **k: None
_pyl_imread = cv2.imread
def _pyl_imread2(path, *a, **k):
    img = _pyl_imread(path, *a, **k)
    if img is None and not os.path.exists(str(path)):
        print('(找不到', path, ':課程 repo 沒附這張圖,改用範例圖片)')
        img = _pyl_sample()
    return img
cv2.imread = _pyl_imread2
print('已改用範例圖片代替攝影機')

### 前置作業
使用前要先到 Teachable Machine 訓練模型,匯出成 Keras 格式,把 keras_model.h5 和 labels.txt 放在這支程式旁邊。

In [ ]:
# 03 — 載入 Teachable Machine 匯出的模型，接上 webcam 做即時分類

### ⚠️ 環境變數一定要先設
TF_USE_LEGACY_KERAS 設成 1,讓新版 TensorFlow 也能讀 Teachable Machine 匯出的舊格式模型。重點是它必須寫在 import tensorflow 之前,順序錯了就沒效。

In [ ]:
#
# 前置作業：
# 1) 到 https://teachablemachine.withgoogle.com/ 訓練你的模型（見 02 教學）
# 2) Export → Tensorflow → Keras → 下載 .zip → 解壓縮
# 3) 把 keras_model.h5 和 labels.txt 放到跟這支程式同一個資料夾
#
# 需要：pip install tensorflow opencv-python numpy

import os
# 必須在 import tensorflow 之前設定：讓 TF 2.16+ 也能吃 TM 匯出的舊格式 h5
os.environ["TF_USE_LEGACY_KERAS"] = "1"

import cv2
import numpy as np
import tensorflow as tf
from PIL import Image, ImageDraw, ImageFont

### 模型與標籤的路徑
一樣用程式所在的資料夾組出模型檔和標籤檔的路徑。

In [ ]:
BASE = os.path.dirname(os.path.abspath(__file__))
MODEL_PATH  = os.path.join(BASE, "keras_model.h5")

### 用 PIL 畫中文
OpenCV 的 putText 不支援中文,所以找一個中文字型,用 Pillow 畫字:先把 BGR 轉成 RGB 給 Pillow 畫,畫完再轉回 BGR。

In [ ]:
LABELS_PATH = os.path.join(BASE, "labels.txt")

# ====== 中文字型（OpenCV putText 不支援中文，改用 PIL）======
FONT_CANDIDATES = [
    "C:/Windows/Fonts/msjh.ttc",
    "C:/Windows/Fonts/msyh.ttc",
    "C:/Windows/Fonts/simhei.ttf",
]
FONT_PATH = next((p for p in FONT_CANDIDATES if os.path.exists(p)), None)
if FONT_PATH is None:
    raise RuntimeError("找不到中文字型")

### 讀類別名稱
labels.txt 每一行長得像「0 空格 類別名稱」。split 加上 1,代表只在第一個空格切一刀,名稱裡就算有空格也不會被切壞。

In [ ]:
def 畫中文(img_bgr, text, xy, size, color_bgr):
    img_pil = Image.fromarray(cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB))
    draw = ImageDraw.Draw(img_pil)
    font = ImageFont.truetype(FONT_PATH, size)
    draw.text(xy, text, font=font, fill=(color_bgr[2], color_bgr[1], color_bgr[0]))
    return cv2.cvtColor(np.array(img_pil), cv2.COLOR_RGB2BGR)

# ====== 讀 labels ======
# TM 的 labels.txt 每一行格式：'0 類別名稱'
with open(LABELS_PATH, "r", encoding="utf-8") as f:
    labels = [line.strip().split(" ", 1)[1] for line in f if line.strip()]

### 載入模型
load_model 讀進模型。compile 等於 False,表示我們只拿來推論,不需要訓練用的設定。印出輸入大小,通常是 224 乘 224 乘 3。

In [ ]:
print(f"讀到 {len(labels)} 個類別：{labels}")

# ====== 讀模型 ======
# compile=False 表示只要拿來推論，不需要 loss / optimizer
model = tf.keras.models.load_model(MODEL_PATH, compile=False)

### 開鏡頭
打開 0 號鏡頭,提醒按 q 離開。

In [ ]:
print("模型載入成功，輸入大小：", model.input_shape)

# ====== 開 webcam ======
cap = cv2.VideoCapture(0)
if not cap.isOpened():
    raise RuntimeError("無法開啟 webcam")

print("按 q 離開")

### 前處理:224、RGB、負一到一
每一格畫面都要整理成模型訓練時看到的樣子:縮成 224 乘 224、轉成 RGB、除以 127.5 再減 1,讓數值落在負一到一之間,最後再多加一個批次維度。

### 推論:取機率最大的類別
predict 回傳每一類的機率,加起來是 1。argmax 找出機率最大的那一類,就是模型的答案。

### 畫結果與機率長條
左上角用大字寫出答案和信心百分比。下面每一類畫一條長條,長度和機率成正比。

### 顯示與收拾
顯示畫面,按 q 離開,最後放掉鏡頭、關閉視窗。

In [ ]:
while True:
    ret, frame = cap.read()
    if not ret:
        break

    # ====== 預處理：TM 的模型固定要 224x224、RGB、[-1, 1] 正規化 ======
    img = cv2.resize(frame, (224, 224))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB).astype(np.float32)
    img = (img / 127.5) - 1                # normalize 到 [-1, 1]
    img = np.expand_dims(img, axis=0)      # 加上 batch 維度 → (1, 224, 224, 3)

    # ====== 推論 ======
    preds = model.predict(img, verbose=0)[0]   # shape = (類別數,)
    top_idx = int(np.argmax(preds))
    top_label = labels[top_idx]
    top_conf  = float(preds[top_idx])

    # ====== 畫在鏡頭畫面上（中文用 PIL）======
    frame = 畫中文(frame, f"{top_label}  {top_conf*100:.1f}%",
                   (10, 10), 32, (0, 255, 0))

    # 底下：每一類的機率長條圖
    for i, (name, prob) in enumerate(zip(labels, preds)):
        y = 70 + i * 30
        bar_w = int(prob * 300)
        cv2.rectangle(frame, (10, y), (10 + bar_w, y + 22), (0, 200, 255), -1)
        frame = 畫中文(frame, f"{name}: {prob*100:5.1f}%",
                       (320, y - 2), 20, (0, 0, 0))

    cv2.imshow("Teachable Machine 分類", frame)

    if cv2.waitKey(1) & 0xFF == ord("q"):
        break

cap.release()
cv2.destroyAllWindows()

## 小結
你把網頁訓練出的模型接到真實鏡頭上了。下一支會打開黑箱,自己從零寫一個 CNN。

👉 回到 [pyl Academy 這堂課](https://friencdking0000.github.io/pyl-academy/#/learn/PY-CVAI/4/0) 看逐句拆解、做練習。